# Venmito Data Diagnostics

Checking for problems in files in `data/`.

**Dependencies: none.** Pure Python 3 standard library — no pandas, no PyYAML — so it runs
anywhere and you can audit the parsing logic rather than trusting a library's coercions.

Each section prints a `CHECK` line with `PASS` / `FAIL`, where `PASS` means *the claimed
problem was reproduced*. A `FAIL` means the claim did not hold up.

**Section index**
1. Load the five files
2. People split across two files with incompatible schemas
3. Join keys are inconsistent across files
4. `promotions.csv` — duplicate primary keys
5. `transactions.xml` — arithmetic and structural problems
6. Planted / synthetic records
7. `transfers.csv` — nulls and outliers
8. The id-join caveat (what is *not* proven)
9. Summary table

In [1]:
import json, csv, re, collections, datetime, os
from xml.etree import ElementTree as ET

# Point this at your data directory if the notebook lives elsewhere.
DATA = os.environ.get("VENMITO_DATA", "data")
if not os.path.isdir(DATA) and os.path.isdir("../data"):
    DATA = "../data"
print("Reading from:", os.path.abspath(DATA))
print(sorted(os.listdir(DATA)))

RESULTS = []  # (section, claim, passed)
def check(section, claim, passed, detail=""):
    RESULTS.append((section, claim, passed))
    print(f"CHECK [{section}] {'PASS' if passed else 'FAIL'} :: {claim}")
    if detail:
        print("        " + detail)

Reading from: /Users/herminiobodon/venmito-template/data
['people.json', 'people.yml', 'promotions.csv', 'transactions.xml', 'transfers.csv']


## 1. Load the five files

In [2]:
def load_json_people():
    with open(os.path.join(DATA, "people.json")) as fh:
        return json.load(fh)

def load_yml_people():
    """Minimal parser for this file's flat 'list of scalar maps' shape.

    Deliberately not PyYAML: keeps the notebook dependency-free AND makes the
    quoting/format quirks visible instead of silently normalised away.
    """
    recs, cur = [], None
    with open(os.path.join(DATA, "people.yml")) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith("- "):
                cur = {}
                recs.append(cur)
                line = " " + line[1:]
            m = re.match(r"\s+([A-Za-z_]+):\s*(.*)$", line)
            if m and cur is not None:
                cur[m.group(1)] = m.group(2)
    return recs

def unquote(v):
    return v.strip().strip('"') if isinstance(v, str) else v

J_raw = load_json_people()
Y_raw = load_yml_people()
P = list(csv.DictReader(open(os.path.join(DATA, "promotions.csv"))))
T = list(csv.DictReader(open(os.path.join(DATA, "transfers.csv"))))
TX = list(ET.parse(os.path.join(DATA, "transactions.xml")).getroot())

print(f"people.json      {len(J_raw):>5} records")
print(f"people.yml       {len(Y_raw):>5} records")
print(f"promotions.csv   {len(P):>5} rows")
print(f"transfers.csv    {len(T):>5} rows")
print(f"transactions.xml {len(TX):>5} transactions")

people.json        933 records
people.yml         297 records
promotions.csv     236 rows
transfers.csv      614 rows
transactions.xml   189 transactions


In [3]:
# Index both people files by integer id.
J = {int(p["id"]): p for p in J_raw}
Y = {int(unquote(r["id"])): r for r in Y_raw}
print("json ids sample:", [p["id"] for p in J_raw[:3]], "-> parsed", sorted(J)[:3])
print("yml  ids sample:", [r["id"] for r in Y_raw[:3]], "-> parsed", sorted(Y)[:3])

json ids sample: ['0001', '0002', '0003'] -> parsed [1, 2, 3]
yml  ids sample: ['1', '3', '5'] -> parsed [1, 3, 5]


## 2. People split across two files with incompatible schemas

Claim: the two files overlap on 228 ids, union is exactly 1002 people with **no gaps**,
neither file alone is the population, and every field uses a different representation.

In [4]:
jids, yids = set(J), set(Y)
overlap, union = jids & yids, jids | yids
missing = sorted(set(range(1, max(union) + 1)) - union)

print(f"json only : {len(jids - yids)}")
print(f"yml  only : {len(yids - jids)}")
print(f"overlap   : {len(overlap)}")
print(f"union     : {len(union)}   id range {min(union)}..{max(union)}")
print(f"gaps      : {missing}")

check("2", "overlap is exactly 228 ids", len(overlap) == 228, f"got {len(overlap)}")
check("2", "union is 1002 people, ids 1..1002 with no gaps",
      len(union) == 1002 and not missing and min(union) == 1 and max(union) == 1002)
check("2", "neither file alone is the population",
      len(jids) < 1002 and len(yids) < 1002, f"json={len(jids)} yml={len(yids)}")

json only : 705
yml  only : 69
overlap   : 228
union     : 1002   id range 1..1002
gaps      : []
CHECK [2] PASS :: overlap is exactly 228 ids
        got 228
CHECK [2] PASS :: union is 1002 people, ids 1..1002 with no gaps
CHECK [2] PASS :: neither file alone is the population
        json=933 yml=297


In [5]:
# Field-representation differences, shown side by side for one person in both files.
pid = sorted(overlap)[0]
print("person id", pid)
print(" json:", json.dumps(J[pid], indent=2))
print(" yml :", json.dumps(Y[pid], indent=2))

print()
print(f"{'concept':<10} {'people.json':<34} {'people.yml'}")
print("-" * 78)
rows = [
    ("id",       'zero-padded string "%s"' % J[pid]["id"],        'int %s' % Y[pid]["id"]),
    ("name",     "first_name + last_name (2 fields)",             'name: "%s"' % Y[pid]["name"]),
    ("location", "nested {City, Country}",                        'city: "%s"' % Y[pid]["city"]),
    ("devices",  "array %s" % J[pid]["devices"],                  "3 x 0/1 columns"),
    ("dob",      "%s  (MM/DD/YYYY)" % J[pid]["dob"],              "%s  (long form)" % Y[pid]["dob"]),
    ("phone",    "key 'telephone'",                               "key 'phone'"),
]
for a, b, c in rows:
    print(f"{a:<10} {b:<34} {c}")

person id 1
 json: {
  "id": "0001",
  "first_name": "Jamie",
  "last_name": "Bright",
  "telephone": "533-849-3913",
  "email": "Jamie.Bright@example.com",
  "devices": [
    "Android"
  ],
  "location": {
    "City": "Montreal",
    "Country": "Canada"
  },
  "dob": "05/20/2000"
}
 yml : {
  "Android": "1",
  "Desktop": "0",
  "Iphone": "0",
  "city": "Montreal, Canada",
  "email": "Jamie.Bright@example.com",
  "id": "1",
  "name": "Jamie Bright",
  "phone": "533-849-3913",
  "dob": "May 20, 2000"
}

concept    people.json                        people.yml
------------------------------------------------------------------------------
id         zero-padded string "0001"          int 1
name       first_name + last_name (2 fields)  name: "Jamie Bright"
location   nested {City, Country}             city: "Montreal, Canada"
devices    array ['Android']                  3 x 0/1 columns
dob        05/20/2000  (MM/DD/YYYY)           May 20, 2000  (long form)
phone      key 'telephone'      

In [6]:
# Do the 228 overlapping records actually agree? Diff every field.
def yml_devices(r):
    return {d for d in ("Android", "Desktop", "Iphone") if unquote(r[d]) == "1"}

def yml_dob_to_json_fmt(v):
    v = unquote(v)
    for fmt in ("%B %d, %Y", "%m/%d/%Y"):
        try:
            return datetime.datetime.strptime(v, fmt).strftime("%m/%d/%Y")
        except ValueError:
            pass
    return "UNPARSEABLE:" + v

disagree = collections.defaultdict(list)
for i in sorted(overlap):
    y, j = Y[i], J[i]
    if unquote(y["name"]) != f"{j['first_name']} {j['last_name']}":
        disagree["name"].append((i, unquote(y["name"]), f"{j['first_name']} {j['last_name']}"))
    if unquote(y["phone"]) != j["telephone"]:
        disagree["phone"].append((i, unquote(y["phone"]), j["telephone"]))
    if unquote(y["email"]).lower() != j["email"].lower():
        disagree["email"].append((i, unquote(y["email"]), j["email"]))
    if unquote(y["city"]) != f"{j['location']['City']}, {j['location']['Country']}":
        disagree["city"].append((i, unquote(y["city"]), j["location"]))
    if yml_dob_to_json_fmt(y["dob"]) != j["dob"]:
        disagree["dob"].append((i, unquote(y["dob"]), j["dob"]))
    if yml_devices(y) != set(j["devices"]):
        disagree["devices"].append((i, sorted(yml_devices(y)), sorted(j["devices"])))

for field, rows in disagree.items():
    print(f"{field:<9} {len(rows)} disagreement(s): {rows[:3]}")

conflicting_ids = {r[0] for rows in disagree.values() for r in rows}
print("\nids with ANY conflict:", sorted(conflicting_ids))
check("2", "overlapping records agree except for exactly one id (998)",
      conflicting_ids == {998}, f"conflicting ids = {sorted(conflicting_ids)}")

name      1 disagreement(s): [(998, 'Fatimah Johns', 'Fey Kuser')]
phone     1 disagreement(s): [(998, '520-433-7411', '999-999-9999')]
email     1 disagreement(s): [(998, 'Fatimah.Johns@example.com', 'fey_kuser@example.com')]
city      1 disagreement(s): [(998, 'London, United Kingdom', {'City': 'Santa Claus', 'Country': 'USA'})]
dob       1 disagreement(s): [(998, 'March 01, 1987', '01/01/1970')]
devices   1 disagreement(s): [(998, ['Iphone'], ['Android', 'Desktop', 'Iphone'])]

ids with ANY conflict: [998]
CHECK [2] PASS :: overlapping records agree except for exactly one id (998)
        conflicting ids = [998]


In [7]:
# Geography: the split is not random, so using one file drops whole countries.
jc = collections.Counter(p["location"]["Country"] for p in J.values())
yc = collections.Counter(unquote(r["city"]).split(", ")[-1] for r in Y.values())
print("people.json countries:", dict(jc))
print("people.yml  countries:", dict(yc))

only_in_yml = set(yc) - set(jc)
print("countries absent from people.json:", sorted(only_in_yml))
check("2", "France and Spain exist only in people.yml",
      {"France", "Spain"} <= only_in_yml)

people.json countries: {'Canada': 157, 'USA': 705, 'United Kingdom': 71}
people.yml  countries: {'Canada': 157, 'France': 42, 'United Kingdom': 71, 'Spain': 26, 'USA': 1}
countries absent from people.json: ['France', 'Spain']
CHECK [2] PASS :: France and Spain exist only in people.yml


## 3. Join keys are inconsistent across files

- `transfers.csv` → person **id**
- `transactions.xml` → **phone**
- `promotions.csv` → **email OR phone** (never both missing)

In [8]:
print("transfers.csv    columns:", list(T[0].keys()))
print("promotions.csv   columns:", list(P[0].keys()))
print("transactions.xml tags   :", [c.tag for c in TX[0]])

transfers.csv    columns: ['sender_id', 'recipient_id', 'amount', 'date']
promotions.csv   columns: ['id', 'client_email', 'telephone', 'promotion', 'responded', 'promotion_date']
transactions.xml tags   : ['items', 'phone', 'store', 'date']


In [9]:
# Build lookups from the MERGED people set (not one file).
phone_to_id, email_to_id = {}, {}
for i, j in J.items():
    phone_to_id[j["telephone"]] = i
    email_to_id[j["email"].lower()] = i
for i, y in Y.items():
    phone_to_id.setdefault(unquote(y["phone"]), i)
    email_to_id.setdefault(unquote(y["email"]).lower(), i)
print(f"{len(phone_to_id)} distinct phones, {len(email_to_id)} distinct emails across 1002 people")

1002 distinct phones, 1002 distinct emails across 1002 people


In [10]:
# promotions: how contactable is each row?
no_email = sum(1 for r in P if not r["client_email"].strip())
no_phone = sum(1 for r in P if not r["telephone"].strip())
no_contact = sum(1 for r in P if not r["client_email"].strip() and not r["telephone"].strip())
print(f"rows with no email : {no_email}")
print(f"rows with no phone : {no_phone}")
print(f"rows with NEITHER  : {no_contact}")
check("3", "promotions has 69 email-less and 85 phone-less rows",
      (no_email, no_phone) == (69, 85), f"got {(no_email, no_phone)}")
check("3", "no promotion row is entirely uncontactable", no_contact == 0)

unresolved = [r for r in P
              if (r["client_email"].strip().lower() not in email_to_id)
              and (r["telephone"].strip() not in phone_to_id)]
check("3", "every promotion resolves to a real person via email OR phone",
      not unresolved, f"{len(unresolved)} unresolved")

rows with no email : 69
rows with no phone : 85
rows with NEITHER  : 0
CHECK [3] PASS :: promotions has 69 email-less and 85 phone-less rows
        got (69, 85)
CHECK [3] PASS :: no promotion row is entirely uncontactable
CHECK [3] PASS :: every promotion resolves to a real person via email OR phone
        0 unresolved


In [ ]:
#Observe customers with no phone and/or email:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200, "display.width", 120)

DATA = os.environ.get("VENMITO_DATA", "data")
if not os.path.isdir(DATA) and os.path.isdir("../data"):
    DATA = "../data"

# keep_default_na=False keeps blanks as "" rather than NaN, so "missing" stays a
# single concept instead of two (NaN vs empty string) that you must test separately.
promos = pd.read_csv(os.path.join(DATA, "promotions.csv"), dtype=str, keep_default_na=False)
for col in ("client_email", "telephone"):
    promos[col] = promos[col].str.strip()

no_email = promos["client_email"].eq("")
no_phone = promos["telephone"].eq("")

promos["contact_status"] = np.select(
    [no_email & no_phone, no_email, no_phone],
    ["neither", "phone only", "email only"],
    default="both",
)

# --- summary -------------------------------------------------------------
print(f"{len(promos)} promotion rows\n")
print(promos["contact_status"].value_counts().rename_axis("contact_status").to_string())
print(f"\nmissing email        : {no_email.sum()}")
print(f"missing phone        : {no_phone.sum()}")
print(f"missing at least one : {(no_email | no_phone).sum()}")
print(f"missing BOTH         : {(no_email & no_phone).sum()}   <- 0 means every row is still resolvable")

# --- the rows themselves -------------------------------------------------
incomplete = promos.loc[no_email | no_phone].copy()
print(f"\n{len(incomplete)} rows missing at least one contact field:")
display(incomplete[["id", "client_email", "telephone", "promotion", "responded",
                    "promotion_date", "contact_status"]])

236 promotion rows

contact_status
email only    85
both          82
phone only    69

missing email        : 69
missing phone        : 85
missing at least one : 154
missing BOTH         : 0   <- 0 means every row is still resolvable

154 rows missing at least one contact field:


,id,client_email,telephone,promotion,responded,promotion_date,contact_status
1,2,,885-134-1740,RedCow,No,2022-01-21,phone only
2,3,,870-909-4612,Dovee,Yes,2023-12-29,phone only
3,4,Dulcie.Melendez@example.com,,RedCow,Yes,2022-08-22,email only
4,5,,561-365-7378,Popsi,Yes,2023-08-10,phone only
5,6,fey_kuser@example.com,,Flixnet,Yes,2024-01-21,email only
6,7,Chester.Bell@example.com,,Flixnet,No,2022-07-06,email only
7,8,Cecilia.Edwards@example.com,,Popsi,No,2024-01-17,email only
8,9,,958-724-3846,Oreoz,Yes,2022-08-03,phone only
11,12,,176-108-4411,Flixnet,No,2023-03-15,phone only
12,13,Annalise.Stephens@example.com,,Dovee,No,2022-06-03,email only


In [11]:
# transactions: orphan phones
tx_phones = {e.findtext("phone") for e in TX}
orphans = sorted(tx_phones - set(phone_to_id))
print("transaction phones not present in either people file:", orphans)
for e in TX:
    if e.findtext("phone") in set(orphans):
        print(f"  tx id={e.get('id'):<5} phone={e.findtext('phone')} store={e.findtext('store')}")
check("3", "exactly 4 transaction phones are orphans", len(orphans) == 4, str(orphans))

transaction phones not present in either people file: ['505-325-3117', '570-105-4278', '777-777-7777', '888-888-8888']
  tx id=3003  phone=570-105-4278 store=SparkMart
  tx id=3021  phone=505-325-3117 store=PetPals Mart
  tx id=4002  phone=777-777-7777 store=SparkMart
  tx id=4003  phone=888-888-8888 store=BestChoice Buy
CHECK [3] PASS :: exactly 4 transaction phones are orphans
        ['505-325-3117', '570-105-4278', '777-777-7777', '888-888-8888']


## 4. `promotions.csv` — duplicate primary keys

Claim: ids 200–212 each appear twice on completely unrelated rows.

In [12]:
idc = collections.Counter(r["id"] for r in P)
dupe_ids = sorted((k for k, v in idc.items() if v > 1), key=int)
print("duplicated ids:", dupe_ids)
print(f"{len(P)} rows but only {len(idc)} distinct ids")

print("\nid 200 appears as:")
for r in P:
    if r["id"] == "200":
        print("  ", r)

check("4", "ids 200-212 are each duplicated",
      dupe_ids == [str(i) for i in range(200, 213)], str(dupe_ids))
check("4", "the duplicate-id rows are unrelated records (not copies)",
      len({tuple(r.values()) for r in P if r["id"] == "200"}) == 2)

duplicated ids: ['200', '201', '202', '203', '204', '205', '206', '207', '208', '209', '210', '211', '212']
236 rows but only 223 distinct ids

id 200 appears as:
   {'id': '200', 'client_email': 'Bibi.Roberts@example.com', 'telephone': '', 'promotion': 'Krafty Cheddar', 'responded': 'Yes', 'promotion_date': '2023-05-23'}
   {'id': '200', 'client_email': 'Dean.Lewis@example.com', 'telephone': '243-955-3782', 'promotion': 'Flixnet', 'responded': 'Yes', 'promotion_date': '2023-01-15'}
CHECK [4] PASS :: ids 200-212 are each duplicated
        ['200', '201', '202', '203', '204', '205', '206', '207', '208', '209', '210', '211', '212']
CHECK [4] PASS :: the duplicate-id rows are unrelated records (not copies)


In [13]:
# Separately: the same person targeted twice with the same promotion.
pair = collections.Counter((r["client_email"], r["telephone"], r["promotion"]) for r in P)
repeats = [(k, v) for k, v in pair.items() if v > 1]
for k, v in repeats:
    print(f"  x{v}  {k}")
check("4", "4 genuine duplicate (contact, promotion) pairs", len(repeats) == 4, f"got {len(repeats)}")

  x3  ('fey_kuser@example.com', '', 'Flixnet')
  x2  ('fey_kuser@example.com', '', 'Colgatex')
  x2  ('', '492-337-4468', 'Flixnet')
  x3  ('fey_kuser@example.com', '', 'Oreoz')
CHECK [4] PASS :: 4 genuine duplicate (contact, promotion) pairs
        got 4


## 5. `transactions.xml` — arithmetic and structural problems

In [15]:
# 5a. The confusing nesting: <item> contains a child ALSO named <item>.
print(ET.tostring(TX[0], encoding="unicode").strip())
print()
naive = len(TX[0].findall(".//item"))
real  = len(TX[0].find("items"))
print(f"naive .//item count on transaction 1 : {naive}")
print(f"actual line-item count               : {real}")
check("5", "naive //item XPath double-counts line items", naive == 2 * real)

<transaction id="1">
        <items>
            <item>
                <item>GatorBoost</item>
                <price>3</price>
                <price_per_item>3</price_per_item>
                <quantity>1</quantity>
            </item>
        </items>
        <phone>245-506-5389</phone>
        <store>PetPals Mart</store>
        <date>2024-04-12</date>
    </transaction>

naive .//item count on transaction 1 : 2
actual line-item count               : 1
CHECK [5] PASS :: naive //item XPath double-counts line items


In [16]:
# 5b. price != price_per_item * quantity
bad_math, negative = [], []
for e in TX:
    for it in e.find("items"):
        nm = it.findtext("item")
        p, ppi, q = float(it.findtext("price")), float(it.findtext("price_per_item")), float(it.findtext("quantity"))
        if abs(ppi * q - p) > 0.011:
            bad_math.append((e.get("id"), nm, p, ppi, q))
        if p < 0:
            negative.append((e.get("id"), nm, p))

print(f"{'tx id':<7} {'item':<14} {'price':>8} {'ppi':>6} {'qty':>5} {'expected':>9}")
for tid, nm, p, ppi, q in bad_math:
    print(f"{tid:<7} {nm:<14} {p:>8} {ppi:>6} {q:>5} {ppi*q:>9}")
print("\nnegative prices:", negative)

check("5", "exactly 2 rows where price != price_per_item * quantity", len(bad_math) == 2)
check("5", "transaction 4003 has a negative price", negative == [("4003", "Flixnet", -50.0)])

tx id   item              price    ppi   qty  expected
4002    GatorBoost          0.0    3.0  10.0      30.0
4003    Flixnet           -50.0   10.0   1.0      10.0

negative prices: [('4003', 'Flixnet', -50.0)]
CHECK [5] PASS :: exactly 2 rows where price != price_per_item * quantity
CHECK [5] PASS :: transaction 4003 has a negative price


In [17]:
# 5c. Exact duplicate transaction.
def signature(e):
    return (e.findtext("phone"), e.findtext("store"), e.findtext("date"),
            tuple(sorted((it.findtext("item"), it.findtext("quantity"), it.findtext("price"))
                         for it in e.find("items"))))

sigs = collections.defaultdict(list)
for e in TX:
    sigs[signature(e)].append(e.get("id"))
dupe_tx = {s: ids for s, ids in sigs.items() if len(ids) > 1}
for s, ids in dupe_tx.items():
    print("ids", ids, "->", s)
check("5", "transactions 5000 and 5001 are byte-identical duplicates",
      list(dupe_tx.values()) == [["5000", "5001"]])

ids ['5000', '5001'] -> ('416-772-8288', 'SparkMart', '2023-08-25', (('GatorBoost', '2', '6'), ('Oreoz', '6', '18')))
CHECK [5] PASS :: transactions 5000 and 5001 are byte-identical duplicates


In [18]:
# 5d. Unstable unit prices: same product, many price_per_item values.
prices = collections.defaultdict(set)
for e in TX:
    for it in e.find("items"):
        prices[it.findtext("item")].add(float(it.findtext("price_per_item")))
for nm in sorted(prices, key=lambda n: -len(prices[n])):
    vals = sorted(prices[nm])
    print(f"{nm:<16} {len(vals)} distinct: {vals}")
check("5", "no product has a single stable unit price",
      all(len(v) > 1 for v in prices.values()))

Flixnet          7 distinct: [1.0, 5.0, 10.0, 11.0, 12.0, 25.0, 30.0]
GatorBoost       5 distinct: [1.0, 2.0, 3.0, 4.0, 5.0]
Colgatex         5 distinct: [1.0, 2.0, 3.0, 4.0, 6.0]
Oreoz            5 distinct: [1.0, 2.0, 3.0, 4.0, 5.0]
Dovee            4 distinct: [1.0, 2.0, 3.0, 4.0]
KittyKat         4 distinct: [1.0, 2.0, 3.0, 4.0]
Krafty Cheddar   4 distinct: [1.0, 4.0, 5.0, 6.0]
Popsi            4 distinct: [1.0, 3.0, 4.0, 5.0]
RedCow           3 distinct: [3.0, 4.0, 5.0]
Coca-Splash      3 distinct: [1.0, 2.0, 3.0]
CHECK [5] PASS :: no product has a single stable unit price


In [19]:
# 5e. Non-contiguous ids: do not assume density or ordering.
ids = [int(e.get("id")) for e in TX]
print(f"{len(ids)} transactions, id range {min(ids)}..{max(ids)}")
print("ids >= 3000:", sorted(i for i in ids if i >= 3000))
check("5", "id range vastly exceeds the row count", max(ids) > 25 * len(ids))

189 transactions, id range 1..5001
ids >= 3000: [3000, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009, 3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019, 3020, 3021, 3022, 4002, 4003, 5000, 5001]
CHECK [5] PASS :: id range vastly exceeds the row count


## 6. Planted / synthetic records

The `fey_kuser` easter egg occupies two ids and overwrites a real person.

In [20]:
print("people.json id 998:", json.dumps(J[998]))
print("people.yml  id 998:", json.dumps(Y[998]))
print("people.yml  id 1002:", json.dumps(Y[1002]))

check("6", "id 998 is a different person in each file",
      unquote(Y[998]["name"]) != f"{J[998]['first_name']} {J[998]['last_name']}",
      f"yml={unquote(Y[998]['name'])!r} vs json={J[998]['first_name']} {J[998]['last_name']}")
check("6", "Fey Kuser occupies both id 998 (json) and id 1002 (yml)",
      "Fey Kuser" in (f"{J[998]['first_name']} {J[998]['last_name']}",) and
      unquote(Y[1002]["name"]) == "Fey Kuser")
check("6", "the two Fey Kuser records disagree on phone and email",
      unquote(Y[1002]["phone"]) != J[998]["telephone"] and
      unquote(Y[1002]["email"]).lower() != J[998]["email"].lower())

people.json id 998: {"id": "0998", "first_name": "Fey", "last_name": "Kuser", "telephone": "999-999-9999", "email": "fey_kuser@example.com", "devices": ["Android", "Iphone", "Desktop"], "location": {"City": "Santa Claus", "Country": "USA"}, "dob": "01/01/1970"}
people.yml  id 998: {"Android": "0", "Desktop": "0", "Iphone": "1", "city": "London, United Kingdom", "email": "Fatimah.Johns@example.com", "id": "998", "name": "Fatimah Johns", "phone": "520-433-7411", "dob": "March 01, 1987"}
people.yml  id 1002: {"Android": "1", "Desktop": "1", "Iphone": "1", "city": "\"Santa Claus, USA\"", "email": "\"fey.kuser@example.com\"", "id": "1002", "name": "\"Fey Kuser\"", "phone": "\"999-999-1111\"", "dob": "\"01/01/1970\""}
CHECK [6] PASS :: id 998 is a different person in each file
        yml='Fatimah Johns' vs json=Fey Kuser
CHECK [6] PASS :: Fey Kuser occupies both id 998 (json) and id 1002 (yml)
CHECK [6] PASS :: the two Fey Kuser records disagree on phone and email


In [21]:
# id 1002 breaks the YAML file's own conventions: quoted values + a different dob format.
quoted = [i for i, r in Y.items() if any(str(v).strip().startswith('"') for v in r.values())]
print("yml records using quoted values:", quoted)

def dob_style(v):
    v = unquote(v)
    return "slash" if re.fullmatch(r"\d{2}/\d{2}/\d{4}", v) else "long"

styles = collections.Counter(dob_style(r["dob"]) for r in Y.values())
print("yml dob styles:", dict(styles))
print("yml records using the slash style:", [i for i, r in Y.items() if dob_style(r["dob"]) == "slash"])
check("6", "id 1002 is the only quoted yml record", quoted == [1002])
check("6", "id 1002 is the only yml record with a non-long dob format",
      [i for i, r in Y.items() if dob_style(r["dob"]) == "slash"] == [1002])

yml records using quoted values: [1002]
yml dob styles: {'long': 296, 'slash': 1}
yml records using the slash style: [1002]
CHECK [6] PASS :: id 1002 is the only quoted yml record
CHECK [6] PASS :: id 1002 is the only yml record with a non-long dob format


In [22]:
# The 1002 -> 998 transfer drumbeat.
fey = [r for r in T if r["sender_id"] == "1002" and r["recipient_id"] == "998"]
amounts = {r["amount"] for r in fey}
dates = sorted(datetime.date.fromisoformat(r["date"]) for r in fey)
gaps = collections.Counter((b - a).days for a, b in zip(dates, dates[1:]))
non_null = [r for r in T if r["sender_id"].strip()]
share = len(fey) / len(non_null)

print(f"count      : {len(fey)}")
print(f"amounts    : {amounts}")
print(f"date range : {dates[0]} .. {dates[-1]}")
print(f"day gaps   : {dict(gaps)}")
print(f"share of all non-null transfers: {share:.1%}")
check("6", "55 identical weekly 20.00 transfers from 1002 to 998",
      len(fey) == 55 and amounts == {"20.00"} and set(gaps) == {7})
check("6", "that single pair is ~9% of all transfers", 0.08 < share < 0.10, f"{share:.1%}")

count      : 55
amounts    : {'20.00'}
date range : 2023-04-01 .. 2024-04-13
day gaps   : {7: 54}
share of all non-null transfers: 9.2%
CHECK [6] PASS :: 55 identical weekly 20.00 transfers from 1002 to 998
CHECK [6] PASS :: that single pair is ~9% of all transfers
        9.2%


In [23]:
# It dominates any naive "top sender" ranking.
print("top senders by transfer count:")
for sid, n in collections.Counter(r["sender_id"] for r in non_null).most_common(5):
    print(f"  id {sid:<6} {n}")

top senders by transfer count:
  id 1002   55
  id 979    7
  id 986    7
  id 960    6
  id 878    5


## 7. `transfers.csv` — nulls and outliers

In [24]:
# 7a. Fully empty rows, clustered on the 16th of the month.
empties = [r for r in T if not r["sender_id"].strip()]
print(f"{len(empties)} fully empty rows")
for r in empties:
    print("  ", r)
days = {datetime.date.fromisoformat(r["date"]).day for r in empties}
print("\ndistinct day-of-month:", days)
exact_dupes = [k for k, v in collections.Counter(tuple(r.values()) for r in empties).items() if v > 1]
print("exact duplicates among them:", exact_dupes)
check("7", "15 fully empty rows exist", len(empties) == 15)
check("7", "they all fall on the 16th of a month", days == {16})
check("7", "all amounts on empty rows are 0.00",
      {r["amount"] for r in empties} == {"0.00"})

15 fully empty rows
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-03-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-03-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-03-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-03-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-03-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-04-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-04-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-04-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-04-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-04-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-05-16'}
   {'sender_id': '', 'recipient_id': '', 'amount': '0.00', 'date': '2023-05-16'}
   {'sen

In [25]:
# 7b. Self-transfer.
self_tx = [r for r in T if r["sender_id"].strip() and r["sender_id"] == r["recipient_id"]]
print("self-transfers:", self_tx)
check("7", "exactly one self-transfer (945 -> 945)",
      len(self_tx) == 1 and self_tx[0]["sender_id"] == "945")

self-transfers: [{'sender_id': '945', 'recipient_id': '945', 'amount': '78.81', 'date': '2022-08-19'}]
CHECK [7] PASS :: exactly one self-transfer (945 -> 945)


In [26]:
# 7c. Amount distribution and outliers.
amts = sorted(float(r["amount"]) for r in non_null)
n = len(amts)
print(f"n={n}  min={amts[0]}  median={amts[n//2]}  p95={amts[int(n*0.95)]}  max={amts[-1]}")

outliers = [r for r in non_null if float(r["amount"]) > 200]
print(f"\n{len(outliers)} transfers above 200:")
for r in sorted(outliers, key=lambda r: r["date"]):
    print(f"  {r['date']}  {r['sender_id']:>4} -> {r['recipient_id']:<4} {float(r['amount']):>10,.2f}")

ones = [r for r in non_null if float(r["amount"]) == 1.00]
print(f"\ntransfers of exactly 1.00: {len(ones)}")
check("7", "9 transfers exceed 200 against a typical range of ~1-100", len(outliers) == 9)
check("7", "11 transfers of exactly 1.00", len(ones) == 11)

n=599  min=0.84  median=27.24  p95=94.26  max=15000.0

9 transfers above 200:
  2023-06-15   777 -> 888      999.99
  2023-06-16   888 -> 777      999.99
  2023-06-17   777 -> 888      999.99
  2023-06-18   888 -> 777      999.99
  2023-06-19   777 -> 888      999.99
  2023-08-01   123 -> 456    5,000.00
  2023-08-01   123 -> 789    5,000.00
  2023-08-01   123 -> 321    5,000.00
  2023-09-30   654 -> 123   15,000.00

transfers of exactly 1.00: 11
CHECK [7] PASS :: 9 transfers exceed 200 against a typical range of ~1-100
CHECK [7] PASS :: 11 transfers of exactly 1.00


In [27]:
# 7d. The two suspicious patterns, isolated.
print("wash-trade pattern (777 <-> 888, consecutive days, identical amount):")
for r in sorted((r for r in non_null if {r["sender_id"], r["recipient_id"]} == {"777", "888"}),
                key=lambda r: r["date"]):
    print("  ", r)

print("\nfan-out pattern (single sender, same day, round amounts):")
for r in non_null:
    if r["sender_id"] == "123" and float(r["amount"]) >= 5000:
        print("  ", r)
print("\nand the inbound leg:")
for r in non_null:
    if r["recipient_id"] == "123" and float(r["amount"]) >= 5000:
        print("  ", r)

wash-trade pattern (777 <-> 888, consecutive days, identical amount):
   {'sender_id': '777', 'recipient_id': '888', 'amount': '999.99', 'date': '2023-06-15'}
   {'sender_id': '888', 'recipient_id': '777', 'amount': '999.99', 'date': '2023-06-16'}
   {'sender_id': '777', 'recipient_id': '888', 'amount': '999.99', 'date': '2023-06-17'}
   {'sender_id': '888', 'recipient_id': '777', 'amount': '999.99', 'date': '2023-06-18'}
   {'sender_id': '777', 'recipient_id': '888', 'amount': '999.99', 'date': '2023-06-19'}

fan-out pattern (single sender, same day, round amounts):
   {'sender_id': '123', 'recipient_id': '456', 'amount': '5000.00', 'date': '2023-08-01'}
   {'sender_id': '123', 'recipient_id': '789', 'amount': '5000.00', 'date': '2023-08-01'}
   {'sender_id': '123', 'recipient_id': '321', 'amount': '5000.00', 'date': '2023-08-01'}

and the inbound leg:
   {'sender_id': '654', 'recipient_id': '123', 'amount': '15000.00', 'date': '2023-09-30'}


In [28]:
# 7e. Referential integrity: every non-null id resolves to a real person.
unknown = {int(r[f]) for r in non_null for f in ("sender_id", "recipient_id")
           if int(r[f]) not in union}
print("transfer ids not found in the merged people set:", unknown or "none")
check("7", "no referential-integrity failures among non-null transfer ids", not unknown)

transfer ids not found in the merged people set: none
CHECK [7] PASS :: no referential-integrity failures among non-null transfer ids


## 8. The id-join caveat — what is *not* proven

`transfers.csv` carries **no phone, email, or name**. So the id join is inferred from the
id *domain* (every value lands inside 1..1002 with no orphans), not confirmed by any second
attribute. A transfer extract generated against a different id sequence would look
identical and still be wrong.

The cell below quantifies how strong that circumstantial evidence actually is.

In [29]:
tids = {int(r[f]) for r in non_null for f in ("sender_id", "recipient_id")}
coverage = len(tids & union) / len(tids)
print(f"distinct ids appearing in transfers : {len(tids)}")
print(f"of those, present in people          : {len(tids & union)}  ({coverage:.1%})")
print(f"people id space                      : {min(union)}..{max(union)}")
print(f"transfer id space                    : {min(tids)}..{max(tids)}")
print()
print("What this establishes : the id spaces are compatible; there are no orphan ids.")
print("What it does NOT      : that person N in people.* is the SAME person as sender N.")
print("                        No shared attribute exists in transfers.csv to cross-check.")
check("8", "id-domain evidence is total (100% coverage) but circumstantial", coverage == 1.0)

distinct ids appearing in transfers : 578
of those, present in people          : 578  (100.0%)
people id space                      : 1..1002
transfer id space                    : 1..1002

What this establishes : the id spaces are compatible; there are no orphan ids.
What it does NOT      : that person N in people.* is the SAME person as sender N.
                        No shared attribute exists in transfers.csv to cross-check.
CHECK [8] PASS :: id-domain evidence is total (100% coverage) but circumstantial


In [30]:
# Two hand-verifiable examples, present in BOTH people files.
counts = collections.Counter()
for r in non_null:
    counts[int(r["sender_id"])] += 1
    counts[int(r["recipient_id"])] += 1

examples = [i for i, c in counts.most_common()
            if i in J and i in Y and 4 <= c <= 12 and i not in (998, 1002)][:2]

for pid in examples:
    print("=" * 70)
    print("PERSON ID", pid)
    print("  json:", json.dumps(J[pid]))
    print("  yml :", json.dumps(Y[pid]))
    rows = [r for r in non_null
            if str(pid) in (r["sender_id"], r["recipient_id"])]
    print(f"  transfers ({len(rows)}):")
    for r in rows:
        direction = "sent    " if r["sender_id"] == str(pid) else "received"
        other = r["recipient_id"] if r["sender_id"] == str(pid) else r["sender_id"]
        print(f"    {r['date']}  {direction} {float(r['amount']):>8,.2f}  counterparty {other}")

PERSON ID 108
  json: {"id": "0108", "first_name": "Mohsin", "last_name": "Tucker", "telephone": "174-389-8757", "email": "Mohsin.Tucker@example.com", "devices": ["Desktop"], "location": {"City": "Toronto", "Country": "Canada"}, "dob": "02/17/1999"}
  yml : {"Android": "0", "Desktop": "1", "Iphone": "0", "city": "Toronto, Canada", "email": "Mohsin.Tucker@example.com", "id": "108", "name": "Mohsin Tucker", "phone": "174-389-8757", "dob": "February 17, 1999"}
  transfers (6):
    2023-01-18  received    23.77  counterparty 620
    2023-08-15  sent         3.86  counterparty 592
    2023-10-23  sent        35.97  counterparty 592
    2023-12-04  received     2.89  counterparty 754
    2024-01-02  sent        24.27  counterparty 130
    2024-03-05  received    72.63  counterparty 966
PERSON ID 123
  json: {"id": "0123", "first_name": "Finnian", "last_name": "Fisher", "telephone": "382-879-9946", "email": "Finnian.Fisher@example.com", "devices": ["Android", "Iphone"], "location": {"City": "

## 9. Summary

In [31]:
passed = sum(1 for _, _, ok in RESULTS if ok)
print(f"{passed} / {len(RESULTS)} claims reproduced\n")
width = max(len(c) for _, c, _ in RESULTS)
last = None
for section, claim, ok in RESULTS:
    if section != last:
        print(f"\n-- section {section} " + "-" * (width - 8))
        last = section
    print(f"  {'PASS' if ok else 'FAIL':<5} {claim}")

failures = [c for _, c, ok in RESULTS if not ok]
if failures:
    print("\nFAILED CLAIMS (my diagnosis was wrong on these):")
    for c in failures:
        print("  -", c)
else:
    print("\nEvery claim in the diagnosis reproduced against the files on disk.")

33 / 33 claims reproduced


-- section 2 ------------------------------------------------------
  PASS  overlap is exactly 228 ids
  PASS  union is 1002 people, ids 1..1002 with no gaps
  PASS  neither file alone is the population
  PASS  overlapping records agree except for exactly one id (998)
  PASS  France and Spain exist only in people.yml

-- section 3 ------------------------------------------------------
  PASS  promotions has 69 email-less and 85 phone-less rows
  PASS  no promotion row is entirely uncontactable
  PASS  every promotion resolves to a real person via email OR phone
  PASS  exactly 4 transaction phones are orphans

-- section 4 ------------------------------------------------------
  PASS  ids 200-212 are each duplicated
  PASS  the duplicate-id rows are unrelated records (not copies)
  PASS  4 genuine duplicate (contact, promotion) pairs

-- section 5 ------------------------------------------------------
  PASS  naive //item XPath double-counts line items
  PAS

### Cleaning order these findings imply

1. Normalize both people files to one schema (split `name`, split `city`, unify devices to
   a set, parse both dob formats, cast id to int) and **outer-join on id** — do not pick
   one file, or you drop France and Spain entirely.
2. Decide id 998 explicitly: keep Fatimah Johns, treat Fey Kuser as one entity at 1002, or
   quarantine both. Do not let a silent overwrite stand.
3. Build phone→id and email→id lookups from the *merged* people, then resolve transactions
   (phone) and promotions (email|phone). Expect 4 orphan transaction phones.
4. Re-key promotions on row position or a synthetic key; keep the original `id` as a
   source column only.
5. In transactions, recompute `price = price_per_item * quantity`, drop or flag the zero
   and negative rows, and dedupe 5000/5001.
6. In transfers, drop the 15 null rows (but keep the outage dates as a data-quality note),
   flag the self-transfer, and **tag** the outliers rather than deleting them — they are
   the fraud-detection signal, not noise.